# Imports

In [2]:
!pip install torchxrayvision --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 25.3 MB/s eta 0:00:0000:0100:01m


In [3]:
import os
import pickle
import random
import typing

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchxrayvision as xrv

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet50, ResNet50_Weights,
    densenet121, DenseNet121_Weights
)
from torch.cuda.amp import autocast

from tqdm import tqdm

# Configuration

In [4]:
class Config:

    SEED = 42
    IMAGE_SIZE = 224
    NUM_CLASSES = 3
    LABELS = [
        "Atelectasis",
        "Effusion",
        "Pneumothorax"
    ]
    MODEL_ARCH = ""
    BATCH_SIZE = 32
    EPOCHS = 15
    LR = 1e-4
    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    NUM_WORKERS = 4 if DEVICE.type == "cuda" else 2
    PIN_MEMORY = DEVICE.type == "cuda"
    DATA_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
    CSV_PATH = os.path.join(
        DATA_DIR,
        "Data_Entry_2017.csv"
    )
    MODEL_DIR = "models"

os.makedirs(Config.MODEL_DIR, exist_ok=True)

In [5]:
ARTIFACT_DIR = Path("/kaggle/input/datasets/shehabmahmoudhelmy/artifacts")

# Utilities and Toolkits

In [6]:
def set_seed(seed=Config.SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [7]:
def save_model(model, name):

    path = f"{Config.MODEL_DIR}/{name}.pth"
    torch.save(model.state_dict(), path)


def load_model(model, name):

    path = f"{Config.MODEL_DIR}/{name}.pth"
    model.load_state_dict(
        torch.load(path, map_location=Config.DEVICE)
    )

    return model

In [8]:
FIG_PATH = Path() / "figures"
os.makedirs(FIG_PATH, exist_ok=True)

def save_fig(fig_name, folder=None, tight_layout=True, fig_extension="png", dpi=300):
    if folder:
        path = FIG_PATH / folder
        path.mkdir(exist_ok=True)
        filepath = path / f"{fig_name}.png"
    else:
        filepath = FIG_PATH / f"{fig_name}.png"
    
    if tight_layout:
        plt.tight_layout()
        
    plt.savefig(filepath, format=fig_extension, dpi=dpi)

# Core Model Utilities

## Pytorch Dataset

In [9]:
class ChestXrayDataset(Dataset):
    def __init__(self, df, image_index, target_labels, transform=None, is_xrv=False):
        self.df = df.reset_index(drop=True)
        self.image_index = image_index
        self.target_labels = target_labels
        self.transform = transform
        self.is_xrv = is_xrv
        self.image_names = self.df["Image Index"].values
        self.labels = self.df[self.target_labels].values.astype("float32")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image_path = self.image_index[image_name]

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise RuntimeError(f"Failed to load image: {image_path}")

        if not self.is_xrv:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        if self.transform:
            image = self.transform(image)

        label = torch.from_numpy(self.labels[idx])
        return image, label

## Transforms

In [10]:
def get_train_transforms(image_size):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(image_size, scale=(0.9, 1.0)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomRotation(7),
    	transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.98, 1.02)),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

def get_val_transforms(image_size):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])


In [11]:
def get_xrv_train_transforms(image_size=224):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(image_size, scale=(0.9, 1.0)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomRotation(7),
        transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.98, 1.02)),
        transforms.ToTensor(), # Converts to [0.0, 1.0]
        transforms.Lambda(lambda x: (x * 2048.0) - 1024.0) # Maps to [-1024, 1024]
    ])

def get_xrv_val_transforms(image_size=224):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: (x * 2048.0) - 1024.0)
    ])

## DataLoaders

In [12]:
def create_dataloaders(
    df,
    image_index,
    target_labels,
    batch_size,
    image_size,
    pin_memory,
    num_workers=4,
    model_arch=""
):

    is_xrv = model_arch == "chexpert_densenet"

    train_transform = get_xrv_train_transforms(image_size) if is_xrv else get_train_transforms(image_size)
    val_transform = get_xrv_val_transforms(image_size) if is_xrv else get_val_transforms(image_size)
    
    train_dataset = ChestXrayDataset(
        df[df["split"] == "train"],
        image_index,
        target_labels,
        transform=train_transform,
        is_xrv=is_xrv
    )
    
    val_dataset = ChestXrayDataset(
        df[df["split"] == "val"],
        image_index,
        target_labels,
        transform=val_transform,
        is_xrv=is_xrv
    )
        
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_memory,
        num_workers=num_workers
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=num_workers
    )
    
    return train_loader, val_loader

## Model Factory

In [13]:
def get_model(name, num_classes=3, pretrained=True):

    if name == "resnet50":

        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif name == "densenet121":

        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        model = densenet121(weights=weights)

        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )

    elif name == "efficientnet_b0":

        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

    elif name == "chexpert_densenet":
        
        model = xrv.models.DenseNet(weights="densenet121-res224-chex")
        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )
    
    else:
        raise ValueError("Unknown model")

    return model

## Optimizer Factory

In [14]:
def get_optimizer(name, params, lr, wd):
    if name == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    elif name == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)
    elif name == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, weight_decay=wd)

## 
Focal Loss

In [15]:
class FocalLoss(nn.Module):
    """
    Binary Focal Loss for imbalanced classification.
    Reference: 'Focal Loss for Dense Object Detection' (https://arxiv.org/abs/1708.02002)
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        """
        Args:
            alpha (float): Weighting factor for the positive class. 
                           Negative class weight is evaluated as (1 - alpha).
                           Set to a negative value to disable alpha weighting.
            gamma (float): Focusing parameter to down-weight easy examples.
            reduction (str): Specifies the reduction to apply to the output:
                             'none' | 'mean' | 'sum'.
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits (torch.Tensor): Raw, unnormalized scores from the model.
            targets (torch.Tensor): Binary targets (0 or 1), same shape as logits.
            
        Returns:
            torch.Tensor: Computed Focal Loss.
        """
        # Ensure targets are floats for BCE
        targets = targets.to(dtype=logits.dtype)
        
        # Calculate standard BCE with logits (numerically stable)
        bce_loss = F.binary_cross_entropy_with_logits(
            logits, 
            targets, 
            reduction="none"
        )
        
        # Calculate pt: probability of the true class
        pt = torch.exp(-bce_loss)
        
        # Standard focal loss formula
        focal_loss = (1 - pt) ** self.gamma * bce_loss
        
        # Apply alpha class-balancing
        if self.alpha >= 0:
            # alpha_t = alpha * y + (1 - alpha) * (1 - y)
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            focal_loss = alpha_t * focal_loss

        # Apply reduction
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

## Early Stopping

In [16]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True  # Indicates a new best model was found
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False # No improvement

## AUC Compute

In [17]:
def compute_auc(preds: torch.Tensor, targets: torch.Tensor):
    # 1. Move to CPU and detach from graph before converting to numpy
    preds = torch.sigmoid(preds).detach().cpu().numpy()
    targets = targets.detach().cpu().numpy()

    # 2. Use multi_class or check for multi-label
    aucs = roc_auc_score(
        targets,
        preds,
        average=None  # Returns AUC for each class (e.g., Atelectasis, Effusion, etc.)
    )
    return aucs

## Model Engine (Train & Validate)

In [18]:
def train_one_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: typing.Callable,
    device: torch.device,
    scaler: torch.cuda.amp.GradScaler
) -> float:
    
    model.train()
    
    running_loss = 0.0
    total_samples = 0
    
    # Adding a description helps differentiate train progress bars
    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # 2. set_to_none=True is faster and uses less memory than default
        optimizer.zero_grad(set_to_none=True)

        # 3. Updated autocast syntax (the old torch.cuda.amp.autocast is deprecated)
        with torch.autocast(device_type=device.type):
            outputs = model(images)
            loss = loss_fn(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # 4. Weight loss by batch size to account for uneven final batches
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
        
        # Update progress bar with the current batch loss
        pbar.set_postfix(loss=loss.item())

    return running_loss / total_samples

In [19]:
def validate(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    loss_fn: typing.Callable,
    device: torch.device
) -> typing.Tuple[float, torch.Tensor, torch.Tensor]:

    model.eval()
    
    running_loss = 0.0
    total_samples = 0
    
    preds = []
    targets = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Optional: You can also use autocast in validation to speed up inference
            with torch.autocast(device_type=device.type):
                outputs = model(images)
                loss = loss_fn(outputs, labels)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            # .detach() is a good safety habit before moving to CPU
            preds.append(outputs.detach().cpu())
            targets.append(labels.detach().cpu())

    preds = torch.cat(preds)
    targets = torch.cat(targets)

    return running_loss / total_samples, preds, targets

# Master Sweep Set-up (Hyperparameters Search)

## Weights and Biases Set-up

In [20]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kyrillos_nabil (kyrillosnabil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Sweep Configuration

In [21]:
sweep_config = {
    'method': 'bayes', # Bayesian optimization is best for large search spaces
    'metric': {'name': 'val_mean_auc', 'goal': 'maximize'},
    "early_terminate": {
        "type": "hyperband",
        "min_iter": 3,
        "eta": 2
    },
    'parameters': {
        # Architecture & Optimization
        'optimizer': {'values': ['adamw', 'sgd', 'rmsprop']},
        'lr': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-2},
        'weight_decay': {'values': [1e-4, 1e-2, 0.0]},
        
        # Focal Loss Parameters
        'gamma': {'value': 2.0},
        'alpha': {'value': 0.25},
        
        # Training details
        'batch_size': {'value': 32},
        'scheduler': {'value': 'cosine'},
        
        # Image size
        'image_size': {'value': 224}
    }
}


## Densenet121 Sweep Initialization

In [ ]:
# Initialize the sweep
sweep_id = wandb.sweep(sweep_config, entity="kyrillosnabil", project="nih-chest-pathologies")
print("Sweep ID: {}".format(sweep_id))

Create sweep with ID: ns6p7te7
Sweep URL: https://wandb.ai/kyrillosnabil/nih-chest-pathologies/sweeps/ns6p7te7
Sweep ID: ns6p7te7


## Load Artifacts

In [27]:
preprocessed_dataset_path = ARTIFACT_DIR / "prepared_df.pkl"
with open(preprocessed_dataset_path, "rb") as f:
    df = pickle.load(f)

image_index_path = ARTIFACT_DIR /"image_index.pkl"
with open(image_index_path, "rb") as f:
    image_index = pickle.load(f)

In [ ]:
def train_sweep():

    with wandb.init() as run:
        config = run.config
        wandb.config.update({
            "architecture": Config.MODEL_ARCH
        })
        set_seed(Config.SEED)
    
        try:    
            train_loader, val_loader = create_dataloaders(
                df=df,
                image_index=image_index,
                target_labels=Config.LABELS,
                batch_size=config.batch_size,
                image_size=config.image_size,
                pin_memory=Config.PIN_MEMORY,
                num_workers=Config.NUM_WORKERS,
                model_arch=Config.MODEL_ARCH
            )
            
            model = get_model(Config.MODEL_ARCH).to(Config.DEVICE)

            if Config.MODEL_ARCH == "chexpert_densenet":
                model.op_threshs = None

            optimizer = get_optimizer(
                config.optimizer, 
                model.parameters(), 
                config.lr, 
                config.weight_decay
            )
            
            loss_fn = FocalLoss(alpha=config.alpha, gamma=config.gamma)
            scaler = torch.amp.GradScaler(Config.DEVICE)
            es = EarlyStopping(patience=5)
            
            for epoch in range(Config.EPOCHS):
                # Training logic
                train_loss = train_one_epoch(
                    model, train_loader, optimizer, loss_fn, Config.DEVICE, scaler
                )
                
                # Validation logic
                val_loss, preds, targets = validate(
                    model, val_loader, loss_fn, Config.DEVICE
                )
                
                # Metrics
                auc_per_class = compute_auc(preds, targets)
                mean_auc = auc_per_class.mean()

                # Log metrics to W&B
                wandb.log({
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "val_mean_auc": mean_auc,
                    "atelectasis_auc": auc_per_class[0],
                    "effusion_auc": auc_per_class[1],
                    "pneumothorax_auc": auc_per_class[2],
                    "lr": optimizer.param_groups[0]['lr']
                })
                
                print(f"Epoch {epoch}: AUC {mean_auc:.4f}")
                print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
  
                # Check Early Stopping
                is_best = es(val_loss)
                if is_best:
                    # save_model(model, f"{Config.MODEL_ARCH}_epoch{epoch}")
                    pass
                
                if es.early_stop:
                    print(f"Early stopping triggered at epoch {epoch}")
                    break
                    
        except torch.cuda.OutOfMemoryError:
            print(f"OOM Error with batch_size={config.batch_size}, img_size={config.image_size}. Skipping.")
            wandb.log({"OOM_Error": True, "val_mean_auc": 0.0}) # Penalize this run so Bayes learns to avoid it
            torch.cuda.empty_cache()


## Densenet121 Sweep Agent (Sanity Check)

In [25]:
# 6. Execute the sweep agent
wandb.agent(sweep_id, function=train_sweep, count=10) # count = number of trials

wandb: Agent Starting Run: c9gqgwcl with config:
wandb: 	alpha: 0.25
wandb: 	batch_size: 32
wandb: 	gamma: 2
wandb: 	image_size: 224
wandb: 	lr: 7.192688340869363e-05
wandb: 	optimizer: adamw
wandb: 	scheduler: cosine
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 136MB/s]
/tmp/ipykernel_101/2401441799.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 0: AUC 0.8366
Train Loss: 0.0282 | Val Loss: 0.0267


Epoch 1: AUC 0.8575
Train Loss: 0.0258 | Val Loss: 0.0253


...

Epoch 4: AUC 0.8660
Train Loss: 0.0238 | Val Loss: 0.0251


Epoch 5: AUC 0.8659
Train Loss: 0.0233 | Val Loss: 0.0256


Epoch 6: AUC 0.8644
Train Loss: 0.0227 | Val Loss: 0.0260


Epoch 7: AUC 0.8620
Train Loss: 0.0221 | Val Loss: 0.0263


Training:  31%|███       | 577/1875 [04:01<06:07,  3.53it/s, loss=0.0176] wandb: Ctrl + C detected. Stopping sweep.


## Efficientnet_b0 Sweep Initialization

In [32]:
Config.MODEL_ARCH = "efficientnet_b0"
sweep_id_efficientnetb0 = wandb.sweep(
    sweep_config,
    entity="kyrillosnabil",
    project="nih-chest-pathologies"
)

Create sweep with ID: 3nm46pgv
Sweep URL: https://wandb.ai/kyrillosnabil/nih-chest-pathologies/sweeps/3nm46pgv


## Resnet50 Sweep Initialization

In [34]:
Config.MODEL_ARCH = "resnet50"
sweep_id_resnet50 = wandb.sweep(
    sweep_config,
    entity="kyrillosnabil",
    project="nih-chest-pathologies"
)

Create sweep with ID: eeimz8d6
Sweep URL: https://wandb.ai/kyrillosnabil/nih-chest-pathologies/sweeps/eeimz8d6


## CheXpert DenseNet Sweep Initialization

In [22]:
Config.MODEL_ARCH = "chexpert_densenet"
sweep_id_resnet50 = wandb.sweep(
    sweep_config,
    entity="kyrillosnabil",
    project="nih-chest-pathologies"
)

Create sweep with ID: jpn0swjo
Sweep URL: https://wandb.ai/kyrillosnabil/nih-chest-pathologies/sweeps/jpn0swjo


In [ ]:
wandb.agent(sweep_id_resnet50, function=train_sweep, count=1)   

In [35]:
wandb.agent(sweep_id_efficientnetb0, function=train_sweep, count=1)   

wandb: Agent Starting Run: 0yhqv2md with config:
wandb: 	alpha: 0.25
wandb: 	batch_size: 32
wandb: 	gamma: 2
wandb: 	image_size: 224
wandb: 	lr: 0.00018955638860603785
wandb: 	optimizer: sgd
wandb: 	scheduler: cosine
wandb: 	weight_decay: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 212MB/s]
/tmp/ipykernel_100/57010923.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 0: AUC 0.5820
Train Loss: 0.0417 | Val Loss: 0.0340


Epoch 1: AUC 0.6289
Train Loss: 0.0332 | Val Loss: 0.0331


...

Epoch 8: AUC 0.7251
Train Loss: 0.0308 | Val Loss: 0.0310


Epoch 9: AUC 0.7318
Train Loss: 0.0306 | Val Loss: 0.0309


Epoch 10: AUC 0.7385
Train Loss: 0.0305 | Val Loss: 0.0306


Training:  46%|████▌     | 858/1875 [05:22<04:52,  3.48it/s, loss=0.038] 

: 